# Comparador rápido Raw vs MySQL / Data Warehouse

Este notebook compara **una tabla por vez** contra su archivo raw. Editá la celda de configuración, ejecutá en orden y revisá resumen, hashes, muestras y diferencias.

- `TARGET = "mysql"`: compara `data/raw/{tabla}.csv` contra MySQL reconstruido con joins a dimensiones, siguiendo `test_raw_db.py`.
- `TARGET = "dw"`: compara `data/raw/{archivo}.csv` contra `rep_{tabla}` en ClickHouse, siguiendo `test_raw_dw.py`.


In [8]:
# Configuración editable
TARGET = "dw"        # "mysql" o "dw"
TABLE_NAME = "bureau"
METHOD = "hashing"   # "hashing" o "pandas"

# Para comparación rápida, limita filas después del ordenamiento determinista.
# Usá None para comparar todo lo cargado en memoria.
SAMPLE_SIZE = 100_000
CHUNK_SIZE = 100_000
SHOW_SAMPLE_ROWS = 10
MAX_DIFF_ROWS = 20


## Setup

In [9]:
import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import clickhouse_connect
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from IPython.display import display

pd.set_option("future.no_silent_downcasting", True)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("No se pudo encontrar la raíz del repositorio.")


ROOT_DIR = find_repo_root()
DATA_RAW_DIR = ROOT_DIR / "data" / "raw"
KEYS_FILE = ROOT_DIR / "02_database_connections" / "utils" / "keys.json"
DW_TEST_CONFIG_PATH = ROOT_DIR / "03_dw_pipeline" / "tests" / "config.yml"
DW_SRC_CONFIG_PATH = ROOT_DIR / "03_dw_pipeline" / "src" / "config.yml"
ENV_FILE = ROOT_DIR / ".env"

load_dotenv(ENV_FILE)

print(f"Repo root: {ROOT_DIR}")
print(f"Raw data:  {DATA_RAW_DIR}")


Repo root: /home/alberto/Documents/Github/data-architectures
Raw data:  /home/alberto/Documents/Github/data-architectures/data/raw


## Conexiones y configuración

In [10]:
def load_keys():
    if not KEYS_FILE.exists():
        raise FileNotFoundError(f"No se encontró keys.json en {KEYS_FILE}")
    with open(KEYS_FILE, "r", encoding="utf-8") as f:
        keys_list = json.load(f)
    return {item["table_name"].lower(): item for item in keys_list}


def load_yaml(path):
    if not path.exists():
        raise FileNotFoundError(f"No se encontró {path}")
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


KEYS_DATA = load_keys()
DW_TEST_CONFIG = load_yaml(DW_TEST_CONFIG_PATH)
DW_SRC_CONFIG = load_yaml(DW_SRC_CONFIG_PATH)
DW_TABLES = {item["name"].lower(): item for item in DW_TEST_CONFIG["transaction_tables"]}


def get_mysql_engine():
    user = os.getenv("DB_USER", "root")
    password = os.getenv("DB_PASSWORD", "")
    host = os.getenv("DB_HOST", "127.0.0.1")
    port = os.getenv("DB_PORT", "3306")
    database = os.getenv("DB_NAME", "home_credit")
    if not database:
        raise ValueError("La variable DB_NAME no está definida.")
    return create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}")


def get_clickhouse_client():
    host = os.getenv("CLICKHOUSE_HOST", "localhost")
    port = int(os.getenv("CLICKHOUSE_PORT", "8123") or 8123)
    user = os.getenv("CLICKHOUSE_USER", "default")
    password = os.getenv("CLICKHOUSE_PASSWORD", "")
    return clickhouse_connect.get_client(host=host, port=port, username=user, password=password)


print(f"Tablas con metadata MySQL: {len(KEYS_DATA)}")
print(f"Tablas configuradas DW:    {len(DW_TABLES)}")


Tablas con metadata MySQL: 41
Tablas configuradas DW:    7


## Helpers compartidos

In [11]:
NULL_TOKENS = {"", "NA", "N/A", "NULL", "NAN"}


def quote_identifier(identifier):
    return f"`{identifier}`"


def find_raw_csv(table_name, file_name=None):
    if file_name:
        csv_path = DATA_RAW_DIR / file_name
        if csv_path.exists():
            return csv_path
        raise FileNotFoundError(f"No se encontró el archivo raw configurado: {csv_path}")
    for path in DATA_RAW_DIR.glob("*.csv"):
        if path.stem.lower() == table_name.lower():
            return path
    raise FileNotFoundError(f"No se encontró CSV raw para '{table_name}' en {DATA_RAW_DIR}")


def count_csv_rows(csv_path):
    with open(csv_path, "r", encoding="utf-8") as f:
        return max(sum(1 for _ in f) - 1, 0)


def normalize_null_values(df):
    df = df.replace({None: pd.NA}).copy()
    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: pd.NA
            if pd.isna(x) or x is None or (isinstance(x, str) and x.strip().upper() in NULL_TOKENS)
            else x
        )
    return df


def normalize_for_hash(df):
    df = normalize_null_values(df.copy())
    try:
        df = df.convert_dtypes(dtype_backend="numpy_nullable")
    except Exception:
        df = df.convert_dtypes()
    for col in df.select_dtypes(include=["Float64", "float64"]).columns:
        df[col] = df[col].round(4)
        try:
            non_null = df[col].dropna()
            if len(non_null) > 0 and np.all(np.mod(non_null, 1) == 0):
                df[col] = df[col].astype("Int64")
        except Exception:
            pass
    return df


def compute_hash(df):
    sha256 = hashlib.sha256()
    df = normalize_for_hash(df)
    for col in df.columns:
        col_data = df[col].astype(object).fillna("NULL").to_numpy(dtype=str).tobytes()
        sha256.update(col_data)
    return sha256.hexdigest()


def compare_with_pandas(df_csv, df_target):
    left = normalize_null_values(df_csv.copy()).reset_index(drop=True)
    right = normalize_null_values(df_target.copy()).reset_index(drop=True)
    right = right[left.columns]
    pd.testing.assert_frame_equal(left, right, check_dtype=False, check_exact=False, rtol=1e-5, atol=1e-5)


def show_differences(df_csv, df_target, max_rows=20):
    left = normalize_null_values(df_csv.copy()).reset_index(drop=True)
    right = normalize_null_values(df_target.copy()).reset_index(drop=True)
    right = right[left.columns]
    comparable = (left == right) | (left.isna() & right.isna())
    row_mask = ~comparable.all(axis=1)
    diff_indices = list(row_mask[row_mask].index[:max_rows])
    if not diff_indices:
        print("No se encontraron diferencias fila a fila en el subconjunto cargado.")
        return pd.DataFrame()
    rows = []
    for idx in diff_indices:
        for col in [col for col in left.columns if not comparable.loc[idx, col]]:
            rows.append({"row": idx, "column": col, "csv_value": left.loc[idx, col], "target_value": right.loc[idx, col]})
    diff_df = pd.DataFrame(rows)
    display(diff_df)
    return diff_df


def limit_clause(limit):
    return "" if limit is None else f"LIMIT {int(limit)}"


## MySQL: Raw CSV vs base relacional

In [12]:
def get_mysql_table_metadata(table_name):
    metadata = KEYS_DATA.get(table_name.lower())
    if not metadata:
        raise KeyError(f"No hay metadata en keys.json para '{table_name}'.")
    return metadata


def get_mysql_columns(engine, table_name):
    with engine.connect() as conn:
        result = conn.execute(text(f"SELECT * FROM {quote_identifier(table_name)} LIMIT 0"))
        return list(result.keys())


def build_mysql_reconstruction_query(table_name, csv_columns, db_columns, metadata):
    selects = []
    joins = []
    joined_tables = set()
    for col in csv_columns:
        if col in db_columns:
            selects.append(f"t.{quote_identifier(col)}")
            continue
        fk_col = f"{col}_ID"
        if fk_col in db_columns and fk_col in metadata.get("fk", []):
            dim_table = f"dim_{col.lower()}"
            dim_alias = f"d_{col.lower()}"
            selects.append(f"{dim_alias}.{quote_identifier(col)}")
            if dim_table not in joined_tables:
                joins.append(
                    f"LEFT JOIN {quote_identifier(dim_table)} {dim_alias} "
                    f"ON t.{quote_identifier(fk_col)} = {dim_alias}.{quote_identifier(fk_col)}"
                )
                joined_tables.add(dim_table)
        else:
            print(f"[WARN] Columna '{col}' no encontrada en DB ni reconstruible por FK; se compara como NULL.")
            selects.append(f"NULL AS {quote_identifier(col)}")
    return f"SELECT {', '.join(selects)} FROM {quote_identifier(table_name)} t {' '.join(joins)}"


def clean_mysql_special_columns(table_name, df):
    if table_name.lower() == "application_train":
        drop_cols = [col for col in ["ORGANIZATION_TYPE", "ORGANIZATION_TYPE_2"] if col in df.columns]
        if drop_cols:
            return df.drop(columns=drop_cols)
    return df


def prepare_mysql_comparison(table_name):
    table_name_db = table_name.lower()
    csv_path = find_raw_csv(table_name)
    metadata = get_mysql_table_metadata(table_name)
    engine = get_mysql_engine()
    csv_columns_raw = pd.read_csv(csv_path, nrows=0).columns.tolist()
    db_columns = get_mysql_columns(engine, table_name_db)
    base_query = build_mysql_reconstruction_query(table_name_db, csv_columns_raw, db_columns, metadata)
    pk_cols = metadata.get("pk", [])
    if not pk_cols:
        raise ValueError(f"PK no encontrada para '{table_name}'.")
    sort_cols = pk_cols + [col for col in csv_columns_raw if col not in pk_cols]
    csv_sort_cols = [col for col in sort_cols if col in csv_columns_raw]
    db_sort_cols = [col for col in sort_cols if col in db_columns]
    db_order_clause = ", ".join(f"t.{quote_identifier(col)}" for col in db_sort_cols)
    with engine.connect() as conn:
        target_rows = conn.execute(text(f"SELECT COUNT(*) FROM {quote_identifier(table_name_db)}")).scalar()
        target_columns_raw = pd.read_sql(text(f"{base_query} LIMIT 0"), conn).columns.tolist()
    return {
        "target": "mysql",
        "table_name": table_name_db,
        "csv_path": csv_path,
        "engine": engine,
        "base_query": base_query,
        "csv_columns": clean_mysql_special_columns(table_name_db, pd.DataFrame(columns=csv_columns_raw)).columns.tolist(),
        "target_columns": clean_mysql_special_columns(table_name_db, pd.DataFrame(columns=target_columns_raw)).columns.tolist(),
        "csv_rows": count_csv_rows(csv_path),
        "target_rows": target_rows,
        "csv_sort_cols": csv_sort_cols,
        "target_order_clause": f"ORDER BY {db_order_clause}" if db_order_clause else "",
    }


def load_mysql_frames(ctx, sample_size=None):
    print("Cargando CSV raw y ordenando...")
    df_csv = pd.read_csv(ctx["csv_path"], dtype_backend="numpy_nullable")
    df_csv = clean_mysql_special_columns(ctx["table_name"], df_csv)
    sort_cols = [col for col in ctx["csv_sort_cols"] if col in df_csv.columns]
    df_csv = df_csv.sort_values(by=sort_cols, na_position="first").reset_index(drop=True)
    if sample_size is not None:
        df_csv = df_csv.head(sample_size)
    query = f"{ctx['base_query']} {ctx['target_order_clause']} {limit_clause(sample_size)}"
    print("Consultando MySQL...")
    with ctx["engine"].connect() as conn:
        df_target = pd.read_sql_query(text(query), conn, dtype_backend="numpy_nullable")
    df_target = clean_mysql_special_columns(ctx["table_name"], df_target)
    df_csv = df_csv[df_target.columns]
    return df_csv.reset_index(drop=True), df_target.reset_index(drop=True)


## DW: Raw CSV vs ClickHouse `rep_*`

In [13]:
def get_dw_table_info(table_name):
    table_info = DW_TABLES.get(table_name.lower())
    if not table_info:
        raise KeyError(f"La tabla '{table_name}' no existe en {DW_TEST_CONFIG_PATH}. Disponibles: {sorted(DW_TABLES)}")
    return table_info


def apply_dw_row_filters(df, table_info):
    ref_col = table_info.get("reference_column")
    if ref_col and isinstance(ref_col, str):
        ref_col = [ref_col]
    filter_col = ref_col[0] if ref_col else None
    if "excluded_first_entry" in table_info or "excluded_last_entry" in table_info:
        if not filter_col:
            raise ValueError("reference_column es requerido si se definen exclusiones por rango.")
        if "excluded_first_entry" in table_info and "excluded_last_entry" not in table_info:
            df = df[df[filter_col] < table_info["excluded_first_entry"]]
        elif "excluded_last_entry" in table_info and "excluded_first_entry" not in table_info:
            df = df[df[filter_col] > table_info["excluded_last_entry"]]
        else:
            start = table_info["excluded_first_entry"]
            end = table_info["excluded_last_entry"]
            df = df[~((df[filter_col] >= start) & (df[filter_col] <= end))]
    return df


def count_dw_csv_rows(csv_path, table_info):
    needs_filter = "excluded_first_entry" in table_info or "excluded_last_entry" in table_info
    if not needs_filter:
        return count_csv_rows(csv_path)
    ref_col = table_info.get("reference_column")
    if isinstance(ref_col, list):
        ref_col = ref_col[0]
    df = pd.read_csv(csv_path, usecols=[ref_col], dtype_backend="numpy_nullable")
    return len(apply_dw_row_filters(df, table_info))


def build_clickhouse_order_by(columns):
    order_cols = [f"{quote_identifier(col)} ASC NULLS FIRST" for col in columns]
    return f"ORDER BY {', '.join(order_cols)}" if order_cols else ""


def prepare_dw_comparison(table_name):
    table_info = get_dw_table_info(table_name)
    csv_path = find_raw_csv(table_name, file_name=table_info["file_name"])
    client = get_clickhouse_client()
    storage_db = DW_SRC_CONFIG["databases"]["storage_db"]
    target_table = f"rep_{table_info['name']}"
    if not client.command(f"EXISTS TABLE {storage_db}.{target_table}"):
        raise RuntimeError(f"La tabla {storage_db}.{target_table} no existe en ClickHouse.")
    df_header = pd.read_csv(csv_path, nrows=0)
    excluded_cols = sorted(set(DW_TEST_CONFIG.get("excluded_columns", []) + table_info.get("excluded_columns", [])))
    db_cols_info = client.query(f"DESCRIBE TABLE {storage_db}.{target_table}").result_rows
    target_columns = [row[0] for row in db_cols_info if row[0] not in excluded_cols]
    csv_columns = [col for col in df_header.columns if col not in excluded_cols]
    compare_cols = sorted(csv_columns)
    return {
        "target": "dw",
        "table_name": table_info["name"],
        "csv_path": csv_path,
        "client": client,
        "storage_db": storage_db,
        "target_table": target_table,
        "table_info": table_info,
        "excluded_cols": excluded_cols,
        "csv_columns": csv_columns,
        "target_columns": target_columns,
        "compare_cols": compare_cols,
        "csv_rows": count_dw_csv_rows(csv_path, table_info),
        "target_rows": client.command(f"SELECT count() FROM {storage_db}.{target_table}"),
        "order_by": build_clickhouse_order_by(compare_cols),
    }


def load_dw_frames(ctx, sample_size=None):
    print("Cargando CSV raw, aplicando filtros DW y ordenando...")
    df_csv = pd.read_csv(ctx["csv_path"], dtype_backend="numpy_nullable")
    df_csv = apply_dw_row_filters(df_csv, ctx["table_info"])
    compare_cols = ctx["compare_cols"]
    df_csv = df_csv[compare_cols].sort_values(by=compare_cols, na_position="first").reset_index(drop=True)
    if sample_size is not None:
        df_csv = df_csv.head(sample_size)
    cols_str = ", ".join(quote_identifier(col) for col in compare_cols)
    query = "\n".join([
        f"SELECT {cols_str}",
        f"FROM {ctx['storage_db']}.{ctx['target_table']}",
        ctx["order_by"],
        limit_clause(sample_size),
    ])
    print("Consultando ClickHouse...")
    result = ctx["client"].query(query)
    df_target = pd.DataFrame(result.result_rows, columns=compare_cols)
    df_target = normalize_null_values(df_target)
    return df_csv.reset_index(drop=True), df_target.reset_index(drop=True)


## Ejecutar comparación 1 a 1

In [14]:
if TARGET not in {"mysql", "dw"}:
    raise ValueError("TARGET debe ser 'mysql' o 'dw'.")
if METHOD not in {"hashing", "pandas"}:
    raise ValueError("METHOD debe ser 'hashing' o 'pandas'.")

ctx = prepare_mysql_comparison(TABLE_NAME) if TARGET == "mysql" else prepare_dw_comparison(TABLE_NAME)

schema_pass = ctx["csv_rows"] == ctx["target_rows"]
if TARGET == "mysql":
    schema_pass = schema_pass and ctx["csv_columns"] == ctx["target_columns"]
else:
    schema_pass = schema_pass and sorted(ctx["csv_columns"]) == sorted(ctx["target_columns"])

summary = pd.DataFrame([{
    "target": ctx["target"],
    "table": ctx["table_name"],
    "csv_path": str(ctx["csv_path"].relative_to(ROOT_DIR)),
    "target_table": ctx.get("target_table", ctx["table_name"]),
    "csv_rows": ctx["csv_rows"],
    "target_rows": ctx["target_rows"],
    "csv_columns": len(ctx["csv_columns"]),
    "target_columns": len(ctx["target_columns"]),
    "schema_result": "PASS" if schema_pass else "FAIL",
    "compare_rows": "ALL" if SAMPLE_SIZE is None else SAMPLE_SIZE,
}])
display(summary)

if not schema_pass:
    print("El esquema o la cantidad de filas no coincide. Igual se cargará la muestra para inspección.")
    print("Columnas CSV:", ctx["csv_columns"])
    print("Columnas destino:", ctx["target_columns"])

df_csv_compare, df_target_compare = load_mysql_frames(ctx, SAMPLE_SIZE) if TARGET == "mysql" else load_dw_frames(ctx, SAMPLE_SIZE)

print(f"Filas cargadas para comparar: CSV={len(df_csv_compare)} | destino={len(df_target_compare)}")
print("\nMuestra CSV:")
display(df_csv_compare.head(SHOW_SAMPLE_ROWS))
print("\nMuestra destino:")
display(df_target_compare.head(SHOW_SAMPLE_ROWS))

hash_csv = compute_hash(df_csv_compare)
hash_target = compute_hash(df_target_compare)
hash_pass = hash_csv == hash_target
print("\nHashes del subconjunto cargado:")
print(f"CSV:     {hash_csv}")
print(f"Destino: {hash_target}")
print(f"Resultado hash: {'PASS' if hash_pass else 'FAIL'}")

pandas_pass = None
if METHOD == "pandas" or not hash_pass:
    try:
        compare_with_pandas(df_csv_compare, df_target_compare)
        pandas_pass = True
        print("Resultado pandas: PASS")
    except AssertionError as exc:
        pandas_pass = False
        print("Resultado pandas: FAIL")
        print(str(exc).split("\n")[0])
        show_differences(df_csv_compare, df_target_compare, max_rows=MAX_DIFF_ROWS)

final_result = hash_pass if METHOD == "hashing" else bool(pandas_pass)
print(f"\nResultado final ({METHOD}): {'PASS' if final_result else 'FAIL'}")


,target,table,csv_path,target_table,csv_rows,target_rows,csv_columns,target_columns,schema_result,compare_rows
0,dw,bureau,data/raw/bureau.csv,rep_bureau,1716428,1716428,17,17,PASS,100000


Cargando CSV raw, aplicando filtros DW y ordenando...
Consultando ClickHouse...
Filas cargadas para comparar: CSV=100000 | destino=100000

Muestra CSV:


,AMT_ANNUITY,AMT_CREDIT_MAX_OVERDUE,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CNT_CREDIT_PROLONG,CREDIT_ACTIVE,CREDIT_CURRENCY,CREDIT_DAY_OVERDUE,CREDIT_TYPE,DAYS_CREDIT,DAYS_CREDIT_ENDDATE,DAYS_CREDIT_UPDATE,DAYS_ENDDATE_FACT,SK_ID_BUREAU,SK_ID_CURR
0,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,-5,<NA>,-36,-46.0,5491946,209921
1,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-77,-441.0,5875729,207533
2,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-64,-111.0,6452468,371162
3,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-58,-493.0,5266170,363106
4,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-23,-72.0,6121986,139064
5,<NA>,<NA>,<NA>,185926.5,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-3,<NA>,6388671,425959
6,<NA>,<NA>,<NA>,654993.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-5,<NA>,6008179,405791
7,<NA>,<NA>,<NA>,1204582.5,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,388.0,-5,<NA>,6299480,367465
8,<NA>,<NA>,<NA>,1309572.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-39,<NA>,5078559,375588
9,<NA>,<NA>,<NA>,2250000.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-16,<NA>,6064357,418331



Muestra destino:


,AMT_ANNUITY,AMT_CREDIT_MAX_OVERDUE,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CNT_CREDIT_PROLONG,CREDIT_ACTIVE,CREDIT_CURRENCY,CREDIT_DAY_OVERDUE,CREDIT_TYPE,DAYS_CREDIT,DAYS_CREDIT_ENDDATE,DAYS_CREDIT_UPDATE,DAYS_ENDDATE_FACT,SK_ID_BUREAU,SK_ID_CURR
0,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,-5,<NA>,-36,-46.0,5491946,209921
1,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-77,-441.0,5875729,207533
2,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-64,-111.0,6452468,371162
3,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-58,-493.0,5266170,363106
4,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-23,-72.0,6121986,139064
5,<NA>,<NA>,<NA>,185926.5,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-3,<NA>,6388671,425959
6,<NA>,<NA>,<NA>,654993.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-5,<NA>,6008179,405791
7,<NA>,<NA>,<NA>,1204582.5,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,388.0,-5,<NA>,6299480,367465
8,<NA>,<NA>,<NA>,1309572.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-39,<NA>,5078559,375588
9,<NA>,<NA>,<NA>,2250000.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-16,<NA>,6064357,418331



Hashes del subconjunto cargado:
CSV:     1b334a08d626b2adc0be678122de76c898d970bf27dd13603834eddf6d5e9acb
Destino: 1b334a08d626b2adc0be678122de76c898d970bf27dd13603834eddf6d5e9acb
Resultado hash: PASS

Resultado final (hashing): PASS


## Inspección manual adicional

Después de ejecutar la comparación, `df_csv_compare` y `df_target_compare` quedan disponibles para mirar cortes específicos.


In [15]:
START_ROW = 0
END_ROW = min(20, len(df_csv_compare))

print("CSV")
display(df_csv_compare.iloc[START_ROW:END_ROW])

print("Destino")
display(df_target_compare.iloc[START_ROW:END_ROW])


CSV


,AMT_ANNUITY,AMT_CREDIT_MAX_OVERDUE,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CNT_CREDIT_PROLONG,CREDIT_ACTIVE,CREDIT_CURRENCY,CREDIT_DAY_OVERDUE,CREDIT_TYPE,DAYS_CREDIT,DAYS_CREDIT_ENDDATE,DAYS_CREDIT_UPDATE,DAYS_ENDDATE_FACT,SK_ID_BUREAU,SK_ID_CURR
0,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,-5,<NA>,-36,-46.0,5491946,209921
1,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-77,-441.0,5875729,207533
2,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-64,-111.0,6452468,371162
3,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-58,-493.0,5266170,363106
4,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-23,-72.0,6121986,139064
5,<NA>,<NA>,<NA>,185926.5,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-3,<NA>,6388671,425959
6,<NA>,<NA>,<NA>,654993.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-5,<NA>,6008179,405791
7,<NA>,<NA>,<NA>,1204582.5,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,388.0,-5,<NA>,6299480,367465
8,<NA>,<NA>,<NA>,1309572.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-39,<NA>,5078559,375588
9,<NA>,<NA>,<NA>,2250000.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-16,<NA>,6064357,418331


Destino


,AMT_ANNUITY,AMT_CREDIT_MAX_OVERDUE,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CNT_CREDIT_PROLONG,CREDIT_ACTIVE,CREDIT_CURRENCY,CREDIT_DAY_OVERDUE,CREDIT_TYPE,DAYS_CREDIT,DAYS_CREDIT_ENDDATE,DAYS_CREDIT_UPDATE,DAYS_ENDDATE_FACT,SK_ID_BUREAU,SK_ID_CURR
0,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,-5,<NA>,-36,-46.0,5491946,209921
1,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-77,-441.0,5875729,207533
2,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-64,-111.0,6452468,371162
3,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-58,-493.0,5266170,363106
4,<NA>,<NA>,<NA>,0.0,<NA>,0.0,0,Closed,currency 1,0,Another type of loan,0,<NA>,-23,-72.0,6121986,139064
5,<NA>,<NA>,<NA>,185926.5,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-3,<NA>,6388671,425959
6,<NA>,<NA>,<NA>,654993.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-5,<NA>,6008179,405791
7,<NA>,<NA>,<NA>,1204582.5,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,388.0,-5,<NA>,6299480,367465
8,<NA>,<NA>,<NA>,1309572.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-39,<NA>,5078559,375588
9,<NA>,<NA>,<NA>,2250000.0,<NA>,0.0,0,Active,currency 1,0,Another type of loan,0,<NA>,-16,<NA>,6064357,418331


In [16]:
_ = show_differences(df_csv_compare, df_target_compare, max_rows=MAX_DIFF_ROWS)

,row,column,csv_value,target_value
0,28406,AMT_CREDIT_SUM_DEBT,0.720,0.720
1,28407,AMT_CREDIT_SUM_DEBT,0.720,0.720
2,28421,AMT_CREDIT_SUM_DEBT,25.785,25.785
3,28433,AMT_CREDIT_SUM_DEBT,66.870,66.870
4,28437,AMT_CREDIT_SUM_DEBT,77.625,77.625
5,28441,AMT_CREDIT_SUM_DEBT,95.355,95.355
6,28442,AMT_CREDIT_SUM_DEBT,98.055,98.055
7,28444,AMT_CREDIT_SUM_DEBT,113.490,113.490
8,28448,AMT_CREDIT_SUM_DEBT,131.715,131.715
9,28450,AMT_CREDIT_SUM_DEBT,142.380,142.380
